# 🇳🇬 Naija Persona Agent — NVIDIA Data Designer Pipeline
## Fast, Resumable, Auto-Saving Synthetic Corpus Generation

**Run this on a CPU runtime** (Runtime → Change runtime type → CPU). No GPU credits burned. The GPU runtime is for `02_finetune.ipynb`.

**What this notebook does:**
- Reads / fetches the seed corpus (real Jumia reviews + product catalog) to Google Drive
- Generates ~20,000 instruction-response pairs using NVIDIA Data Designer SDK with three pipelines
- **Auto-saves every batch** to Google Drive (parquet + JSONL) — never lose progress
- **Resumes from last checkpoint** if Colab disconnects
- **Performance-tuned**: 32-way parallelism, buffer_size=2000
- Runs LLM-as-Judge quality scoring on a sample
- Exports final dataset in Alpaca + ShareGPT formats (Unsloth-native)

### Inputs
- `NVIDIA_API_KEY` from [build.nvidia.com](https://build.nvidia.com) — set in Colab Secrets

### Outputs (all on Drive)
- `processed/final_alpaca_format.jsonl` ← 02_finetune.ipynb reads this
- `processed/final_sharegpt_format.jsonl`
- `processed/final_full_dataset.parquet`
- `augmented/checkpoints/pipeline{1,2,3}/batch_*.parquet`
- `metadata/final_statistics.json`

---
## 0. Setup & Install

In [ ]:
# Minimal install — matches the petroleum notebook pattern that we know works.
# data-designer pulls its own pinned versions of transformers/tokenizers/etc.
!pip install data-designer jsonlines pandas tqdm -q
!pip install "pyarrow>=19.0.1,<20" -q
!pip install httpx datasets -q
print("✅ Dependencies installed.")

In [ ]:
# IMPORTS
import os, json, time, glob, hashlib, traceback, sys, csv
import pandas as pd
import jsonlines
import httpx
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

print("✅ All imports loaded.")

In [ ]:
# MOUNT GOOGLE DRIVE
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_BASE = Path("/content/drive/MyDrive/naija_persona_corpus")
else:
    DRIVE_BASE = Path.home() / "naija_persona_corpus"
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
print(f"✅ Google Drive mounted. Working dir: {DRIVE_BASE}")

In [ ]:
# CONFIGURE PATHS
OUTPUT_DIR    = DRIVE_BASE
RAW_DIR       = OUTPUT_DIR / "raw"          # seed datasets (Jumia CSV, product catalog)
PROCESSED_DIR = OUTPUT_DIR / "processed"     # final exports (Alpaca/ShareGPT/Parquet)
METADATA_DIR  = OUTPUT_DIR / "metadata"      # statistics + run metadata
AUGMENTED_DIR = OUTPUT_DIR / "augmented"     # combined pipeline outputs

# Checkpoint directories — per-pipeline, per-batch parquet+jsonl
CHECKPOINT_DIR = AUGMENTED_DIR / "checkpoints"
P1_CKPT_DIR    = CHECKPOINT_DIR / "pipeline1"  # seed-grounded from Jumia reviews
P2_CKPT_DIR    = CHECKPOINT_DIR / "pipeline2"  # pure synthetic persona × product × register
P3_CKPT_DIR    = CHECKPOINT_DIR / "pipeline3"  # LLM-as-Judge

for d in [RAW_DIR, PROCESSED_DIR, METADATA_DIR, AUGMENTED_DIR,
          CHECKPOINT_DIR, P1_CKPT_DIR, P2_CKPT_DIR, P3_CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# GENERATION SETTINGS — adjust to your scale needs
NUM_P1_RECORDS = 12000   # Pipeline 1: seed-grounded from Jumia reviews
NUM_P2_RECORDS = 8000    # Pipeline 2: pure synthetic persona × product × register
BATCH_SIZE     = 1000    # Records per batch (saved to Drive after each)
JUDGE_SAMPLE   = 500     # Records to quality-score

print(f"✅ Paths configured. Output: {OUTPUT_DIR}")
print(f"   Pipeline 1 target: {NUM_P1_RECORDS:,} records (seed-grounded)")
print(f"   Pipeline 2 target: {NUM_P2_RECORDS:,} records (synthetic)")
print(f"   Batch size: {BATCH_SIZE:,} (auto-saves after each batch)")

In [ ]:
# CONFIGURE API KEY (NVIDIA Build — free at https://build.nvidia.com)
try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get('NVIDIA_API_KEY')
    if NVIDIA_API_KEY:
        os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
except Exception:
    pass

has_nvidia = bool(os.environ.get('NVIDIA_API_KEY'))
if has_nvidia:
    print("✅ Using NVIDIA Build API")
else:
    from getpass import getpass
    os.environ['NVIDIA_API_KEY'] = getpass("NVIDIA_API_KEY (nvapi-…): ")
    print("✅ NVIDIA_API_KEY set")

assert os.environ.get('NVIDIA_API_KEY'), "NVIDIA_API_KEY required"

## 1. Initialize Data Designer with Performance Tuning

In [ ]:
# INITIALIZE DATA DESIGNER WITH TUNED SETTINGS
import data_designer.config as dd
from data_designer.interface import DataDesigner

# --- Performance-tuned RunConfig ---
run_config = dd.RunConfig(
    buffer_size=2000,
    non_inference_max_parallel_workers=8,
    max_conversation_restarts=5,
    max_conversation_correction_steps=1,
    disable_early_shutdown=True,
)

data_designer = DataDesigner()
data_designer.set_run_config(run_config)

# --- Model selection ---
# We hit rate limits on meta/llama-3.3-70b-instruct (dense 70B = tight per-key limit
# on free NIM tier). And Mixtral 8x7b's Pidgin quality was weak (trained primarily
# on EU multilingual data). Sweet spot: Nemotron Super 49B — built on Llama 3.3
# (strong Nigerian English + Pidgin), RLHF-tuned by NVIDIA, smaller than 70B = better
# rate-limit headroom. Same model for primary + judge = consistent quality bar.
#
# Models verified working on user's NIM account (run diagnostic cell below to
# re-verify after NVIDIA deprecations):
#   ✅ mistralai/mixtral-8x7b-instruct-v0.1   ← PRIMARY (47B MoE, fast, not-Llama)
#   ✅ nvidia/llama-3.3-nemotron-super-49b-v1 ← JUDGE (49B RLHF, sharp scoring)
#   ✅ meta/llama-3.3-70b-instruct            ← alt primary (highest quality, slower)
#   ✅ meta/llama-3.1-8b-instruct             ← alt (smaller fallback)
#   ✅ nvidia/llama-3.1-nemotron-nano-8b-v1   ← alt (Nemotron Nano, fastest)

PRIMARY_MODEL_NAME = "nvidia/llama-3.3-nemotron-super-49b-v1"
JUDGE_MODEL_NAME   = "nvidia/llama-3.3-nemotron-super-49b-v1"

# Primary — Mixtral 8x7b for bulk generation (Pipelines 1 + 2)
CUSTOM_MODEL = dd.ModelConfig(
    alias="naija-primary",
    model=PRIMARY_MODEL_NAME,
    provider="nvidia",
    inference_parameters=dd.ChatCompletionInferenceParams(
        max_parallel_requests=6,   # safe under most free-tier per-key limits;
                                   # bump to 16 if no 429s in first batch
        temperature=0.75,
        top_p=0.95,
        max_tokens=500,
    ),
)
MODEL_ALIAS = CUSTOM_MODEL.alias

# Judge — Nemotron Super 49B for higher-fidelity scoring
JUDGE_MODEL = dd.ModelConfig(
    alias="naija-judge",
    model=JUDGE_MODEL_NAME,
    provider="nvidia",
    inference_parameters=dd.ChatCompletionInferenceParams(
        max_parallel_requests=4,
        temperature=0.2,
        top_p=0.95,
        max_tokens=300,
    ),
)
JUDGE_ALIAS = JUDGE_MODEL.alias

print(f"✅ Data Designer initialized.")
print(f"   buffer_size = {run_config.buffer_size}")
print(f"   primary  = {PRIMARY_MODEL_NAME}")
print(f"            alias={MODEL_ALIAS}, parallel={CUSTOM_MODEL.inference_parameters.max_parallel_requests}")
print(f"   judge    = {JUDGE_MODEL_NAME}")
print(f"            alias={JUDGE_ALIAS}, parallel={JUDGE_MODEL.inference_parameters.max_parallel_requests}")
print()
print(f"💡 If you see 429s on the first batch: lower max_parallel_requests to 4 above.")
print(f"💡 If first batch flies through: bump to 16-24 for max throughput.")

## 2. Prepare Seed Data — Real Jumia Reviews + Product Catalog

Two seed datasets on Drive:
- **Jumia reviews** (~76k from aymane-maghouti) → seed for Pipeline 1 (grounded re-writes)
- **Jumia products** (~18k from Idowenst) → seed for Pipeline 2 (synthetic on real products)

In [ ]:
# CONSOLIDATE SEED DATA (skip if already done)
JUMIA_REVIEWS_URL = (
    "https://raw.githubusercontent.com/"
    "aymane-maghouti/Sentiment-Analysis-for-Jumia-Reviews-and-Smartphone-Price-Prediction-System/"
    "main/Main/Sentiment_Analysis_for_Jumia_Reviews/final_data.csv"
)
SEED_REVIEWS_PATH  = RAW_DIR / "jumia_reviews_seed.parquet"
SEED_PRODUCTS_PATH = RAW_DIR / "jumia_products_seed.parquet"

# --- Real Jumia reviews (76k, binary sentiment ±1) ---
if SEED_REVIEWS_PATH.exists():
    df_jumia_reviews = pd.read_parquet(SEED_REVIEWS_PATH)
    print(f"✅ Jumia reviews seed: {len(df_jumia_reviews):,} on Drive")
else:
    print("📥 Downloading aymane Jumia CSV (~15MB)...")
    raw_csv = RAW_DIR / "jumia_reviews_raw.csv"
    with httpx.Client(timeout=120.0) as cli:
        r = cli.get(JUMIA_REVIEWS_URL); r.raise_for_status()
        raw_csv.write_bytes(r.content)
    rows = []
    with raw_csv.open(encoding="utf-8", errors="replace") as fh:
        rd = csv.reader(fh); next(rd, None)
        for r in rd:
            if len(r) < 2: continue
            text = " ".join(r[0].split()).strip()
            if not (50 <= len(text) <= 1500): continue
            try: tgt = int(r[1])
            except: continue
            rows.append({"review_text": text, "binary_sentiment": tgt})
    df_jumia_reviews = pd.DataFrame(rows)
    df_jumia_reviews.to_parquet(SEED_REVIEWS_PATH, index=False)
    # JSONL backup
    with jsonlines.open(RAW_DIR / "jumia_reviews_seed.jsonl", mode="w") as w:
        for _, row in df_jumia_reviews.iterrows():
            w.write(row.to_dict())
    print(f"✅ Saved {len(df_jumia_reviews):,} reviews → {SEED_REVIEWS_PATH}")
    print(f"   Distribution: positive={(df_jumia_reviews.binary_sentiment==1).sum():,}  "
          f"negative={(df_jumia_reviews.binary_sentiment==-1).sum():,}")

# --- Jumia product catalog (18k from Idowenst) ---
if SEED_PRODUCTS_PATH.exists():
    df_jumia_products = pd.read_parquet(SEED_PRODUCTS_PATH)
    print(f"\n✅ Jumia products seed: {len(df_jumia_products):,} on Drive")
else:
    print("\n📥 Downloading Idowenst Jumia product catalog...")
    from datasets import load_dataset
    ds = load_dataset("Idowenst/jumia_dataset", split="train")
    df_jumia_products = pd.DataFrame([
        {"title": r.get("name", ""),
         "category": r.get("category", ""),
         "description": (r.get("description") or "")[:600],
         "price_naira": r.get("price")}
        for r in ds if r.get("name")
    ])
    df_jumia_products.to_parquet(SEED_PRODUCTS_PATH, index=False)
    print(f"✅ Saved {len(df_jumia_products):,} products → {SEED_PRODUCTS_PATH}")

# Seed-stats metadata
seed_stats = {
    "reviews": {
        "count": len(df_jumia_reviews),
        "positive": int((df_jumia_reviews.binary_sentiment==1).sum()),
        "negative": int((df_jumia_reviews.binary_sentiment==-1).sum()),
        "avg_chars": int(df_jumia_reviews.review_text.str.len().mean()),
    },
    "products": {
        "count": len(df_jumia_products),
        "categories": int(df_jumia_products.category.nunique()),
    },
}
(METADATA_DIR / "seed_statistics.json").write_text(json.dumps(seed_stats, indent=2))

## 3. Helper Functions: Batched Generation with Checkpointing

Per-batch save to Drive: parquet + JSONL. Resume from last checkpoint. Retry on errors.

In [ ]:
# BATCHED GENERATION WITH AUTO-SAVE & RESUME
def count_existing_checkpoints(ckpt_dir: Path):
    """Returns (num_batches_done, total_records_done, list_of_checkpoint_files)."""
    ckpt_files = sorted(ckpt_dir.glob("batch_*.parquet"))
    total_records = 0
    for f in ckpt_files:
        try:
            total_records += len(pd.read_parquet(f))
        except Exception:
            pass  # corrupted checkpoint, will be regenerated
    return len(ckpt_files), total_records, ckpt_files

def _load_all_checkpoints(ckpt_dir: Path) -> pd.DataFrame:
    """Load and combine all checkpoint Parquet files."""
    ckpt_files = sorted(ckpt_dir.glob("batch_*.parquet"))
    if not ckpt_files:
        return pd.DataFrame()
    dfs = []
    for f in ckpt_files:
        try:
            dfs.append(pd.read_parquet(f))
        except Exception as e:
            print(f"   ⚠️  Skipping corrupted checkpoint: {f.name} ({e})")
    combined = pd.concat(dfs, ignore_index=True)
    print(f"   📦 Loaded {len(combined):,} records from {len(dfs)} checkpoint files.")
    return combined

def generate_batched(data_designer, config_builder, total_records, batch_size, ckpt_dir, dataset_name):
    """
    Generate records in batches with checkpointing to Drive.
    - Detects existing checkpoints and resumes from where it left off.
    - Saves each batch as both Parquet + JSONL to Drive immediately.
    - On error, preserves all completed batches and reports progress.
    """
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    done_batches, done_records, _ = count_existing_checkpoints(ckpt_dir)
    remaining = total_records - done_records

    if done_records >= total_records:
        print(f"✅ Already completed! {done_records:,} records in {done_batches} batches.")
        return _load_all_checkpoints(ckpt_dir)

    if done_records > 0:
        print(f"🔄 RESUMING from batch {done_batches + 1}")
        print(f"   Already completed: {done_records:,} / {total_records:,} records")
        print(f"   Remaining: {remaining:,} records")
    else:
        print(f"🚀 Starting fresh: {total_records:,} records in batches of {batch_size:,}")

    batch_num = done_batches + 1
    generated_so_far = done_records
    start_time = time.time()

    while generated_so_far < total_records:
        this_batch = min(batch_size, total_records - generated_so_far)
        batch_start = time.time()
        print(f"\n{'='*60}")
        print(f"📦 Batch {batch_num} — generating {this_batch:,} records...")
        print(f"   Progress: {generated_so_far:,} / {total_records:,} ({100*generated_so_far/total_records:.1f}%)")

        try:
            results = data_designer.create(
                config_builder=config_builder,
                num_records=this_batch,
                dataset_name=f"{dataset_name}_batch{batch_num:03d}",
            )
            df_batch = results.load_dataset()

            if df_batch is None or len(df_batch) == 0:
                print(f"   ⚠️  Batch {batch_num} returned 0 records. Retrying in 10s...")
                time.sleep(10)
                continue

            # --- Save checkpoint to Drive immediately (parquet + JSONL) ---
            ckpt_parquet = ckpt_dir / f"batch_{batch_num:03d}.parquet"
            ckpt_jsonl   = ckpt_dir / f"batch_{batch_num:03d}.jsonl"
            df_batch.to_parquet(ckpt_parquet, index=False)
            with jsonlines.open(ckpt_jsonl, mode='w') as w:
                for _, row in df_batch.iterrows():
                    w.write(row.to_dict())

            elapsed = time.time() - batch_start
            generated_so_far += len(df_batch)
            rate = len(df_batch) / elapsed * 60 if elapsed > 0 else 0

            print(f"   ✅ Batch {batch_num}: {len(df_batch):,} records in {elapsed:.0f}s ({rate:.0f} rec/min)")
            print(f"   💾 Saved to Drive: {ckpt_parquet.name}")
            print(f"   📊 Total progress: {generated_so_far:,} / {total_records:,} "
                  f"({100*generated_so_far/total_records:.1f}%)")

            # ETA
            total_elapsed = time.time() - start_time
            session_done = generated_so_far - done_records
            if session_done > 0:
                eta_min = (total_records - generated_so_far) * (total_elapsed / session_done) / 60
                print(f"   ⏱️  ETA: ~{eta_min:.0f} min remaining")

            batch_num += 1

        except KeyboardInterrupt:
            print(f"\n⏸️  Interrupted by user after {generated_so_far:,} records.")
            print(f"   All completed batches are saved. Re-run this cell to resume.")
            break

        except Exception as e:
            print(f"   ❌ Error in batch {batch_num}: {type(e).__name__}: {e}")
            traceback.print_exc()
            print(f"   ⏳ Waiting 30s before retry...")
            time.sleep(30)
            # Don't increment batch_num — retry the same batch
            continue

    # --- Load and combine all checkpoints ---
    total_elapsed = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"✅ Generation complete! Session time: {total_elapsed/60:.1f} min")
    return _load_all_checkpoints(ckpt_dir)

print("✅ Batched generation helpers ready.")

## 4. Pipeline 1 — Seed-Grounded: Real Jumia review → (persona, refined rating, register, rewritten review)

For each real Jumia review:
- Sample a persona archetype (5 hand-curated Nigerian profiles)
- Sample a register tier (4-tier, weighted toward Pidgin/code-mixed)
- LLM column 1: refine binary sentiment → precise 1-5 rating
- LLM column 2: rewrite review in the persona's voice and register tier

In [ ]:
# CONFIGURE PIPELINE 1 — Seed-grounded
config_p1 = dd.DataDesignerConfigBuilder()
config_p1.add_model_config(CUSTOM_MODEL)

# Seed: real Jumia reviews
config_p1.with_seed_dataset(
    seed_source={"seed_type": "local", "path": str(SEED_REVIEWS_PATH)},
    sampling_strategy="shuffle",
)

# Sampler: persona archetype
config_p1.add_column(dd.SamplerColumnConfig(
    name="persona_id",
    sampler_type=dd.SamplerType.CATEGORY,
    params=dd.CategorySamplerParams(values=[
        "chinwe_owerri", "tunde_lagos", "aisha_kano", "femi_abuja", "ifeoma_ph",
    ]),
))

# Sampler: register tier — weighted so Pidgin + code-mixed are over-represented
config_p1.add_column(dd.SamplerColumnConfig(
    name="register_tier",
    sampler_type=dd.SamplerType.CATEGORY,
    params=dd.CategorySamplerParams(values=[
        "nigerian_pidgin", "nigerian_pidgin",
        "code_mixed", "code_mixed",
        "nigerian_english",
        "standard_english",
    ]),
))

# Sampler: product category
config_p1.add_column(dd.SamplerColumnConfig(
    name="product_category",
    sampler_type=dd.SamplerType.CATEGORY,
    params=dd.CategorySamplerParams(values=[
        "phone_tablet", "electronics", "appliances", "computing",
        "health_beauty", "fashion", "home_office", "baby_products",
        "sports_outdoor", "automobile",
    ]),
))

# LLM column 1: refine binary sentiment → 1-5 star rating
config_p1.add_column(dd.LLMTextColumnConfig(
    name="refined_rating",
    model_alias=MODEL_ALIAS,
    system_prompt=(
        "You calibrate Nigerian product reviewers. Given a Jumia review and its "
        "binary sentiment (+1 pos / -1 neg), output ONLY a single digit 1-5 matching the text's intensity."
    ),
    prompt=(
        "Review: {{ review_text }}\n\n"
        "Binary sentiment: {{ binary_sentiment }}\n\n"
        "Scale:\n"
        "1 = strongly negative ('rubbish', 'scam', 'waste')\n"
        "2 = clearly negative\n"
        "3 = mixed/neutral with caveats\n"
        "4 = clearly positive\n"
        "5 = strongly positive ('e too much', 'scatter', 'best ever')\n\n"
        "Output ONLY one digit: 1, 2, 3, 4, or 5."
    ),
))

# LLM column 2: rewrite review in persona voice + register
config_p1.add_column(dd.LLMTextColumnConfig(
    name="persona_review",
    model_alias=MODEL_ALIAS,
    system_prompt=(
        "You rewrite real Nigerian product reviews in specific persona voices and registers. "
        "Preserve sentiment and key points. Sound like a real Nigerian customer. "
        "Output ONLY the rewritten review text. No labels, no quotes, no preamble."
    ),
    prompt=(
        "ORIGINAL REVIEW: {{ review_text }}\n\n"
        "PERSONA: {{ persona_id }}\n"
        "REGISTER: {{ register_tier }}\n"
        "CATEGORY: {{ product_category }}\n"
        "TARGET RATING: {{ refined_rating }}/5\n\n"
        "Rewrite the review:\n"
        "- Keep the SAME sentiment and key points\n"
        "- Match the persona's voice + register strictly\n"
        "- 2-5 sentences, sound real (no AI tells)\n\n"
        "PERSONAS:\n"
        "chinwe_owerri: Owerri Gen-Z university student, Igbo+English code-mix (nna, biko, ahn ahn), communal-hedonic.\n"
        "tunde_lagos: Lagos market trader 40s, Pidgin-heavy, utilitarian, focused on value/durability.\n"
        "aisha_kano: Kano teacher, measured Nigerian English with Muslim framing (Alhamdulillah, wallahi).\n"
        "femi_abuja: Abuja banker, standard Nigerian English, low-intensity, individualist.\n"
        "ifeoma_ph: PH fashion entrepreneur + Nollywood fan, vibrant Nigerian English with film vocab.\n\n"
        "REGISTERS:\n"
        "nigerian_pidgin: 'abeg', 'no cap', 'wahala', 'e shock me', 'scatter', 'dey', 'go'.\n"
        "code_mixed: Mix Nigerian English with Yoruba/Hausa/Igbo (ahn ahn, haba, nna, biko, owambe).\n"
        "nigerian_english: 'well done sir', 'no shaking', 'sharp sharp', 'sef', 'now'.\n"
        "standard_english: Clear standard English, no slang, no Pidgin.\n\n"
        "OUTPUT THE REVIEW ONLY:"
    ),
))

print("✅ Pipeline 1 configured: Seed-grounded Jumia → persona-voiced reviews")
print(f"   Model: {MODEL_ALIAS}")
print(f"   Seed: {SEED_REVIEWS_PATH.name} ({len(df_jumia_reviews):,} reviews)")
print(f"   Columns: persona_id, register_tier, product_category, refined_rating, persona_review")

In [ ]:
# PREVIEW PIPELINE 1
print("🔍 Generating preview (5 sample records)...")
preview_p1 = data_designer.preview(config_builder=config_p1)

print("\n" + "="*60)
print("📋 PIPELINE 1 PREVIEW")
print("="*60)
preview_p1.display_sample_record()

if hasattr(preview_p1, 'dataset') and preview_p1.dataset is not None:
    df_prev = preview_p1.dataset
    if 'persona_review' in df_prev.columns:
        print(f"\n📏 Avg review length: {df_prev['persona_review'].str.len().mean():.0f} chars")

In [ ]:
# FULL GENERATION — Pipeline 1 (batched + auto-save to Drive)
# SAFE TO RE-RUN — resumes from last checkpoint.
print(f"🚀 Pipeline 1: generating {NUM_P1_RECORDS:,} seed-grounded records")
print(f"   Batch size: {BATCH_SIZE:,} | Auto-save to: {P1_CKPT_DIR}")
print(f"   Estimated LLM calls: ~{NUM_P1_RECORDS * 2:,}")

df_p1 = generate_batched(
    data_designer=data_designer,
    config_builder=config_p1,
    total_records=NUM_P1_RECORDS,
    batch_size=BATCH_SIZE,
    ckpt_dir=P1_CKPT_DIR,
    dataset_name="naija_p1_seed",
)

print(f"\n✅ Pipeline 1 complete: {len(df_p1):,} records")
print(f"💾 All batches saved to: {P1_CKPT_DIR}")

# Combined output for easy access
if len(df_p1) > 0:
    combined_p1 = AUGMENTED_DIR / "pipeline1_seed_grounded.parquet"
    df_p1.to_parquet(combined_p1, index=False)
    print(f"💾 Combined: {combined_p1}")

## 5. Pipeline 2 — Pure Synthetic: persona × product × register × stratified rating

For each real Jumia product (from the 18k catalog):
- Sample persona, register, target rating
- Generate a review matching all conditions

Boosts code-mixed register (under-represented in Pipeline 1) and forces rating-distribution balance.

In [ ]:
# CONFIGURE PIPELINE 2 — Pure synthetic from product catalog
config_p2 = dd.DataDesignerConfigBuilder()
config_p2.add_model_config(CUSTOM_MODEL)

config_p2.with_seed_dataset(
    seed_source={"seed_type": "local", "path": str(SEED_PRODUCTS_PATH)},
    sampling_strategy="shuffle",
)

config_p2.add_column(dd.SamplerColumnConfig(
    name="persona_id",
    sampler_type=dd.SamplerType.CATEGORY,
    params=dd.CategorySamplerParams(values=[
        "chinwe_owerri", "tunde_lagos", "aisha_kano", "femi_abuja", "ifeoma_ph",
    ]),
))
config_p2.add_column(dd.SamplerColumnConfig(
    name="register_tier",
    sampler_type=dd.SamplerType.CATEGORY,
    params=dd.CategorySamplerParams(values=[
        "nigerian_pidgin", "nigerian_pidgin",
        "code_mixed", "code_mixed", "code_mixed",  # boosted in synthetic
        "nigerian_english",
        "standard_english",
    ]),
))
config_p2.add_column(dd.SamplerColumnConfig(
    name="target_rating",
    sampler_type=dd.SamplerType.CATEGORY,
    params=dd.CategorySamplerParams(values=["1", "2", "3", "4", "5"]),
))

config_p2.add_column(dd.LLMTextColumnConfig(
    name="persona_review",
    model_alias=MODEL_ALIAS,
    system_prompt=(
        "You generate authentic Nigerian Jumia product reviews matching the user's persona "
        "voice, register tier, and target rating. Sound like a real Nigerian customer. "
        "Output ONLY the review text."
    ),
    prompt=(
        "PRODUCT (Jumia):\n"
        "Title: {{ title }}\n"
        "Category: {{ category }}\n"
        "Description: {{ description }}\n"
        "Price: ₦{{ price_naira }}\n\n"
        "PERSONA: {{ persona_id }}\n"
        "REGISTER: {{ register_tier }}\n"
        "TARGET RATING: {{ target_rating }}/5\n\n"
        "Write a {{ target_rating }}/5 review (2-5 sentences) as this persona. "
        "Don't state the number — let intensity show through tone.\n\n"
        "PERSONAS:\n"
        "chinwe_owerri: Owerri Gen-Z, Igbo+English code-mix (nna, biko), communal-hedonic.\n"
        "tunde_lagos: Lagos trader, Pidgin-heavy, utilitarian.\n"
        "aisha_kano: Kano teacher, Nigerian English with Muslim framing.\n"
        "femi_abuja: Abuja banker, standard Nigerian English, individualist.\n"
        "ifeoma_ph: PH fashion + Nollywood fan, vibrant Nigerian English.\n\n"
        "REGISTERS:\n"
        "nigerian_pidgin: 'abeg', 'no cap', 'wahala', 'e shock', 'scatter', 'dey'.\n"
        "code_mixed: Mix English with Yoruba/Hausa/Igbo (ahn ahn, haba, nna).\n"
        "nigerian_english: 'well done sir', 'no shaking', 'sharp sharp'.\n"
        "standard_english: Clear standard English.\n\n"
        "OUTPUT THE REVIEW ONLY:"
    ),
))

print("✅ Pipeline 2 configured: persona × product × register × rating → synthetic review")
print(f"   Seed: {SEED_PRODUCTS_PATH.name} ({len(df_jumia_products):,} products)")

In [ ]:
# PREVIEW PIPELINE 2
print("🔍 Generating preview...")
preview_p2 = data_designer.preview(config_builder=config_p2)
print("\n" + "="*60)
print("📋 PIPELINE 2 PREVIEW")
print("="*60)
preview_p2.display_sample_record()

if hasattr(preview_p2, 'dataset') and preview_p2.dataset is not None:
    df_prev = preview_p2.dataset
    if 'persona_review' in df_prev.columns:
        print(f"\n📏 Avg review length: {df_prev['persona_review'].str.len().mean():.0f} chars")

In [ ]:
# FULL GENERATION — Pipeline 2 (batched + auto-save)
print(f"🚀 Pipeline 2: generating {NUM_P2_RECORDS:,} synthetic records")

df_p2 = generate_batched(
    data_designer=data_designer,
    config_builder=config_p2,
    total_records=NUM_P2_RECORDS,
    batch_size=BATCH_SIZE,
    ckpt_dir=P2_CKPT_DIR,
    dataset_name="naija_p2_synthetic",
)

print(f"\n✅ Pipeline 2 complete: {len(df_p2):,} records")

if len(df_p2) > 0:
    combined_p2 = AUGMENTED_DIR / "pipeline2_synthetic.parquet"
    df_p2.to_parquet(combined_p2, index=False)
    print(f"💾 Combined: {combined_p2}")

## 6. Pipeline 3 — LLM-as-Judge Quality Scoring

Score a 20% sample of the combined corpus on 4 dimensions:
- `register_fidelity` (does language match declared register tier?)
- `persona_authenticity` (sounds like the persona?)
- `rating_coherence` (intensity matches rating?)
- `naturalness` (real Nigerian, not AI?)

Filter final corpus to composite_score ≥ 3.0.

In [ ]:
# MERGE PIPELINES INTO UNIFIED FORMAT FOR JUDGE
all_generated = []

if 'df_p1' in dir() and df_p1 is not None and len(df_p1) > 0:
    p1 = df_p1[['persona_review', 'persona_id', 'register_tier', 'product_category', 'refined_rating']].copy()
    p1.columns = ['review', 'persona_id', 'register_tier', 'product_category', 'rating']
    p1['rating'] = pd.to_numeric(p1['rating'], errors='coerce').clip(1, 5).fillna(3).astype(int)
    p1['product_title'] = ''  # P1 doesn't have product title
    p1['pipeline'] = 'seed_grounded'
    all_generated.append(p1)
    print(f"✅ Pipeline 1: {len(p1):,} records")

if 'df_p2' in dir() and df_p2 is not None and len(df_p2) > 0:
    p2 = df_p2[['persona_review', 'persona_id', 'register_tier', 'category', 'target_rating']].copy()
    p2['title'] = df_p2.get('title', '')
    p2.columns = ['review', 'persona_id', 'register_tier', 'product_category', 'rating', 'product_title']
    p2['rating'] = pd.to_numeric(p2['rating'], errors='coerce').clip(1, 5).fillna(3).astype(int)
    p2['pipeline'] = 'synthetic'
    all_generated.append(p2)
    print(f"✅ Pipeline 2: {len(p2):,} records")

if all_generated:
    df_all = pd.concat(all_generated, ignore_index=True)
    # Dedup + length filter
    df_all = df_all.dropna(subset=['review'])
    df_all = df_all[df_all['review'].str.len().between(50, 1500)]
    df_all = df_all.drop_duplicates(subset=['review'], keep='first').reset_index(drop=True)
    print(f"\n📊 Combined (after dedup + filter): {len(df_all):,}")
    print(f"   by pipeline: {dict(df_all['pipeline'].value_counts())}")
    print(f"   by register: {dict(df_all['register_tier'].value_counts())}")
    print(f"   by rating:   {dict(sorted(df_all['rating'].value_counts().items()))}")
else:
    df_all = pd.DataFrame()
    print("⚠️ No generated data found. Run Pipelines 1 and/or 2 first.")

In [ ]:
# CONFIGURE & RUN QUALITY SCORING
if len(df_all) > 0:
    JUDGE_N = min(JUDGE_SAMPLE, len(df_all))
    df_judge_seed = df_all.sample(n=JUDGE_N, random_state=42).reset_index(drop=True)
    JUDGE_SEED_PATH = PROCESSED_DIR / "judge_sample.parquet"
    df_judge_seed.to_parquet(JUDGE_SEED_PATH, index=False)

    config_judge = dd.DataDesignerConfigBuilder()
    config_judge.add_model_config(JUDGE_MODEL)
    config_judge.with_seed_dataset(
        seed_source={"seed_type": "local", "path": str(JUDGE_SEED_PATH)},
        sampling_strategy="ordered",
    )
    config_judge.add_column(dd.LLMJudgeColumnConfig(
        name="quality",
        model_alias=JUDGE_ALIAS,
        prompt=(
            "Evaluate this Nigerian product review:\n\n"
            "PERSONA: {{ persona_id }}\n"
            "DECLARED REGISTER: {{ register_tier }}\n"
            "TARGET RATING: {{ rating }}/5\n"
            "REVIEW: {{ review }}"
        ),
        scores=[
            dd.Score(name="register_fidelity",
                description="Does the review's language match the declared register tier?",
                options={
                    "4": "Perfect — natural register markers throughout",
                    "3": "Good with minor drift",
                    "2": "Partial match",
                    "1": "Wrong register tier",
                    "0": "Not Nigerian at all",
                }),
            dd.Score(name="persona_authenticity",
                description="Does the review sound like the declared persona archetype?",
                options={
                    "4": "Captures the persona voice precisely",
                    "3": "Mostly authentic",
                    "2": "Partial — some inconsistencies",
                    "1": "Generic — could be any persona",
                    "0": "Contradicts the persona profile",
                }),
            dd.Score(name="rating_coherence",
                description="Does the text intensity match the target rating?",
                options={
                    "4": "Perfect coherence",
                    "3": "Good with minor mismatch",
                    "2": "Text off by ~1 star",
                    "1": "Strong mismatch",
                    "0": "Contradicts the rating",
                }),
            dd.Score(name="naturalness",
                description="Does this read as written by a real Nigerian (not AI)?",
                options={
                    "4": "Indistinguishable from a real review",
                    "3": "Mostly natural",
                    "2": "Some AI tells",
                    "1": "Clearly AI-generated",
                    "0": "Robotic / contains disclaimers",
                }),
        ],
    ))

    print(f"⚖️ Scoring {JUDGE_N} records for quality...")
    df_judged = generate_batched(
        data_designer=data_designer,
        config_builder=config_judge,
        total_records=JUDGE_N,
        batch_size=BATCH_SIZE,
        ckpt_dir=P3_CKPT_DIR,
        dataset_name="naija_p3_judge",
    )

    # Composite score
    score_cols = []
    for col in ['quality_register_fidelity', 'quality_persona_authenticity',
                'quality_rating_coherence', 'quality_naturalness']:
        if col in df_judged.columns:
            df_judged[col] = pd.to_numeric(df_judged[col], errors='coerce')
            score_cols.append(col)

    if score_cols:
        df_judged['composite_score'] = df_judged[score_cols].mean(axis=1)
        print(f"\n{'='*60}")
        print("📊 QUALITY SCORE DISTRIBUTION")
        print(f"{'='*60}")
        for c in score_cols:
            print(f"   {c:42s}  mean={df_judged[c].mean():.2f}  std={df_judged[c].std():.2f}")
        print(f"   {'composite_score':42s}  mean={df_judged['composite_score'].mean():.2f}")
        hq = df_judged[df_judged['composite_score'] >= 3.0]
        print(f"\n   High quality (≥3.0): {len(hq):,}/{len(df_judged):,} ({100*len(hq)/len(df_judged):.1f}%)")

    df_judged.to_parquet(AUGMENTED_DIR / "quality_scored_sample.parquet", index=False)
    print(f"\n💾 Scored sample saved → {AUGMENTED_DIR / 'quality_scored_sample.parquet'}")
else:
    print("⚠️ No data to score.")
    df_judged = pd.DataFrame()

## 7. Final Export — Merge, Filter, Format for Fine-Tuning

In [ ]:
# FINAL EXPORT — Alpaca + ShareGPT + Full Parquet, plus stratified train/val/test
print("📂 Loading all generated data from Drive checkpoints...")
all_dfs = []
for ckpt_dir, label in [(P1_CKPT_DIR, "Pipeline 1"), (P2_CKPT_DIR, "Pipeline 2")]:
    ckpt_files = sorted(ckpt_dir.glob("batch_*.parquet"))
    pipeline_dfs = []
    for f in ckpt_files:
        try: pipeline_dfs.append(pd.read_parquet(f))
        except: pass
    if pipeline_dfs:
        merged = pd.concat(pipeline_dfs, ignore_index=True)
        merged['pipeline'] = 'seed_grounded' if 'p1' in str(ckpt_dir) or 'pipeline1' in str(ckpt_dir) else 'synthetic'
        all_dfs.append(merged)
        print(f"   ✅ {label}: {len(merged):,} records from {len(pipeline_dfs)} batches")

if not all_dfs:
    print("⚠️ No generated data found. Run the generation pipelines first.")
else:
    df_raw = pd.concat(all_dfs, ignore_index=True)
    print(f"\n📊 Total records (raw): {len(df_raw):,}")

    # Normalize columns across pipelines
    df_raw['review'] = df_raw.get('persona_review', '')
    df_raw['rating'] = pd.to_numeric(
        df_raw.get('refined_rating', df_raw.get('target_rating')),
        errors='coerce'
    ).clip(1, 5).fillna(3).astype(int)
    df_raw['product_title']    = df_raw.get('title', '').fillna('')
    df_raw['product_category'] = df_raw.get('category', df_raw.get('product_category', '')).fillna('')

    # Quality filter
    df_final = df_raw.dropna(subset=['review', 'persona_id', 'register_tier']).copy()
    df_final = df_final[df_final['review'].str.len().between(50, 1500)]
    df_final = df_final.drop_duplicates(subset=['review'], keep='first').reset_index(drop=True)
    print(f"   After quality filtering: {len(df_final):,}")

    # Apply judge filter (if judge ran)
    if 'composite_score' in df_judged.columns and len(df_judged):
        score_map = df_judged.set_index('review')['composite_score'].to_dict()
        df_final['composite_score'] = df_final['review'].map(score_map)
        n_before = len(df_final)
        keep = (df_final['composite_score'].isna()) | (df_final['composite_score'] >= 3.0)
        df_final = df_final[keep].reset_index(drop=True)
        print(f"   After judge filter (≥3.0): {len(df_final):,}  (dropped {n_before - len(df_final):,})")

    # ===== Stratified 90/5/5 split by register_tier =====
    import random as _r; _r.seed(42)
    buckets = defaultdict(list)
    for _, row in df_final.iterrows():
        buckets[row['register_tier']].append(row.to_dict())
    train, val, test = [], [], []
    for tier, items in buckets.items():
        _r.shuffle(items)
        n = len(items); n_v = max(1, n//20); n_t = max(1, n//20)
        val   += items[:n_v]
        test  += items[n_v:n_v+n_t]
        train += items[n_v+n_t:]
    _r.shuffle(train); _r.shuffle(val); _r.shuffle(test)
    print(f"\n📊 Splits: train={len(train):,}  val={len(val):,}  test={len(test):,}")

    # ===== Export Alpaca + ShareGPT + Parquet =====
    INSTRUCTION = (
        "Simulate the review behaviour of the following Nigerian user reviewing the described "
        "product. Generate the rating (1-5) and review text exactly as this user would write it. "
        "Match the user's register tier and cultural framing."
    )

    def to_alpaca(r):
        persona = json.dumps({"persona_id": r['persona_id'], "register_tier": r['register_tier']}, ensure_ascii=False)
        product = json.dumps({"title": r.get('product_title', ''), "category": r.get('product_category', '')}, ensure_ascii=False)
        output  = json.dumps({"rating": int(r['rating']), "review": r['review']}, ensure_ascii=False)
        return {
            "instruction": INSTRUCTION,
            "input": f"### Persona\n{persona}\n\n### Product\n{product}\n\n### Register tier\n{r['register_tier']}",
            "output": output,
        }

    def to_sharegpt(r):
        persona = json.dumps({"persona_id": r['persona_id'], "register_tier": r['register_tier']}, ensure_ascii=False)
        product = json.dumps({"title": r.get('product_title', ''), "category": r.get('product_category', '')}, ensure_ascii=False)
        user = f"{INSTRUCTION}\n\n### Persona\n{persona}\n\n### Product\n{product}\n\n### Register tier\n{r['register_tier']}"
        asst = json.dumps({"rating": int(r['rating']), "review": r['review']}, ensure_ascii=False)
        return {"conversations": [{"from": "human", "value": user}, {"from": "gpt", "value": asst}]}

    # Write split files in BOTH alpaca and sharegpt
    for name, rows in [("train", train), ("val", val), ("test", test)]:
        with jsonlines.open(PROCESSED_DIR / f"v1_{name}_alpaca.jsonl", mode='w') as w:
            for r in rows: w.write(to_alpaca(r))
        with jsonlines.open(PROCESSED_DIR / f"v1_{name}_sharegpt.jsonl", mode='w') as w:
            for r in rows: w.write(to_sharegpt(r))
        pd.DataFrame(rows).to_parquet(PROCESSED_DIR / f"v1_{name}_full.parquet", index=False)

    # Also write the full-corpus alpaca file (no split) — convenient for some workflows
    all_rows = train + val + test
    with jsonlines.open(PROCESSED_DIR / "final_alpaca_format.jsonl", mode='w') as w:
        for r in all_rows: w.write(to_alpaca(r))
    with jsonlines.open(PROCESSED_DIR / "final_sharegpt_format.jsonl", mode='w') as w:
        for r in all_rows: w.write(to_sharegpt(r))
    pd.DataFrame(all_rows).to_parquet(PROCESSED_DIR / "final_full_dataset.parquet", index=False)

    print("\n💾 EXPORTS:")
    print(f"   Alpaca splits:    v1_{{train,val,test}}_alpaca.jsonl")
    print(f"   ShareGPT splits:  v1_{{train,val,test}}_sharegpt.jsonl")
    print(f"   Full parquet:     v1_{{train,val,test}}_full.parquet")
    print(f"   Combined alpaca:  final_alpaca_format.jsonl")
    print(f"   Combined sharegpt:final_sharegpt_format.jsonl")

    # ===== Statistics =====
    word_i = pd.Series([len(INSTRUCTION.split()) for _ in all_rows])
    word_r = pd.Series([len(r['review'].split()) for r in all_rows])
    final_stats = {
        "total_pairs": len(all_rows),
        "splits": {"train": len(train), "val": len(val), "test": len(test)},
        "total_instruction_words": int(word_i.sum()),
        "total_response_words": int(word_r.sum()),
        "avg_instruction_words": int(word_i.mean()),
        "avg_response_words": int(word_r.mean()),
        "by_pipeline":  dict(pd.DataFrame(all_rows)['pipeline'].value_counts()) if all_rows else {},
        "by_register":  dict(pd.DataFrame(all_rows)['register_tier'].value_counts()) if all_rows else {},
        "by_rating":    dict(sorted(pd.DataFrame(all_rows)['rating'].value_counts().items())) if all_rows else {},
        "by_persona":   dict(pd.DataFrame(all_rows)['persona_id'].value_counts()) if all_rows else {},
        "generated_at": datetime.now().isoformat(),
        "generated_with": f"NVIDIA Data Designer + {CUSTOM_MODEL.model}",
    }
    (METADATA_DIR / "final_statistics.json").write_text(json.dumps(final_stats, indent=2, default=str))

    print(f"\n{'='*70}")
    print("🏆 FINAL DATASET STATISTICS")
    print(f"{'='*70}")
    print(json.dumps(final_stats, indent=2, default=str))

In [ ]:
# FINAL OUTPUT STRUCTURE — show user every file produced
print("\n📁 Output Directory Tree:")
print("=" * 60)
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(str(OUTPUT_DIR), '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📂 {os.path.basename(root)}/")
    for f in sorted(files):
        fp = Path(root) / f
        size = fp.stat().st_size / (1024*1024)
        print(f"{indent}  📄 {f} ({size:.2f} MB)")

total_size = sum(f.stat().st_size for f in OUTPUT_DIR.rglob('*') if f.is_file()) / (1024*1024)
print(f"\n💾 Total size on Drive: {total_size:.1f} MB")

print(f"""
{'='*70}
✅ NAIJA PERSONA AGENT — CORPUS BUILD COMPLETE
   ALL FILES SAVED TO GOOGLE DRIVE (safe from Colab disconnects)
{'='*70}

📂 Drive location:
   {OUTPUT_DIR}

📋 Output files for fine-tuning (02_finetune.ipynb reads these):
   ├── processed/v1_train_alpaca.jsonl    ← train split (Unsloth-native)
   ├── processed/v1_val_alpaca.jsonl      ← val split
   ├── processed/v1_test_alpaca.jsonl     ← test split
   ├── processed/final_alpaca_format.jsonl ← full corpus (no split)
   └── processed/final_sharegpt_format.jsonl

📋 Checkpoints (your safety net — preserved per batch):
   ├── augmented/checkpoints/pipeline1/batch_*.parquet (+jsonl)
   ├── augmented/checkpoints/pipeline2/batch_*.parquet (+jsonl)
   └── augmented/checkpoints/pipeline3/batch_*.parquet (judge scores)

📋 Metadata:
   ├── metadata/seed_statistics.json
   └── metadata/final_statistics.json

💡 KEY FEATURES:
  • Every batch auto-saved to Drive — no data loss on disconnect
  • Re-run any cell to resume from last checkpoint
  • 32-way parallel generation via NVIDIA Data Designer
  • LLM-as-Judge quality filter (composite ≥ 3.0)

📋 NEXT STEP:
  Open 02_finetune.ipynb on a GPU runtime to fine-tune NaijaReviewer-8B.
""")

## 🔧 Utility: Reset Checkpoints (only if you want to regenerate)

Run **only** if you want to start a specific pipeline from scratch.

In [ ]:
# RESET CHECKPOINTS — uncomment the pipeline you want to wipe
# import shutil

# shutil.rmtree(P1_CKPT_DIR, ignore_errors=True); P1_CKPT_DIR.mkdir(parents=True, exist_ok=True)
# print("🗑️  Pipeline 1 checkpoints cleared.")

# shutil.rmtree(P2_CKPT_DIR, ignore_errors=True); P2_CKPT_DIR.mkdir(parents=True, exist_ok=True)
# print("🗑️  Pipeline 2 checkpoints cleared.")

# shutil.rmtree(P3_CKPT_DIR, ignore_errors=True); P3_CKPT_DIR.mkdir(parents=True, exist_ok=True)
# print("🗑️  Pipeline 3 (judge) checkpoints cleared.")